# 2.1 ID 怎麼變成可學習的特徵？


[1.6](https://birdhackor.github.io/tiny-perceptron-vlm/1.6.html) 讓每個字查出一整排下一字分數，看到貓就只套用貓那列的規則。如果我們想同時看三個字，並讓它們的資料一起參與計算，就需要另一種中間表示：每個字先查出一小排描述它的數字，後面再把這些數字混合，最後才算候選分數。這小排數字叫特徵向量；向量是一排數，特徵表示它提供後續計算使用的資訊，不一定有「是否動物」這樣的人類名字。

ID 只是字的位置約定，不能直接把貓的ID3乘兩倍當成貓的特徵。我們另外建立一張可調表，每個ID對上一列D個數，叫 embedding（嵌入表示）。字表V個字，表的形狀就是 `[V,D]`；D可以比V小很多。與第1章直接輸出V個候選分數不同，這些D個數還要經過其他層才能回答下一字。

下面先手動指定五列、每列三數，避免把初始亂數誤讀成固定字義。輸入 `[[1,2,1]]` 表示一筆文字、三個位置，第1與第3位置用了同一個ID。tensor軸背景見 [W.3](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

embedding = nn.Embedding(5, 3)
with torch.no_grad():
    embedding.weight.copy_(
        torch.tensor(
            [
                [0.0, 0.0, 0.0],
                [1.0, 0.0, 0.0],
                [0.0, 1.0, 0.0],
                [0.0, 0.0, 1.0],
                [1.0, 1.0, 1.0],
            ]
        )
    )
ids = torch.tensor([[1, 2, 1]])
x = embedding(ids)
print(x.shape)
print(x)
assert torch.equal(x[0, 0], x[0, 2])

`nn.Embedding(5,3)` 建立可學習查表層；`weight` 儲存表格，手動抄入的數字只是本次例子。`embedding(ids)` 對每個位置找出那一列，輸出形狀為 `[1,3,3]`：一筆、三位置、每位置三特徵。三列向量依序是 `[1,0,0]`、`[0,1,0]`、`[1,0,0]`。最後的 `torch.equal` 檢查兩次ID1確實取到完全相同的數值。

下圖左側是同一張五列三欄的表，右側三個位置各查一次。橘色對應ID1，藍色對應ID2；向右箭頭只是查出那列，不是把ID乘成向量。右上與右下同樣得到 `[1,0,0]`，說明第0與第2位置用相同ID，就共用同一列參數。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/foundations_embedding.svg")))

它為什麼叫可學習？因為這張 `weight` 是參數，後面混合特徵、計算答案代價後，梯度能沿 [1.10](https://birdhackor.github.io/tiny-perceptron-vlm/1.10.html) 的運算關係傳回用到的表格列，再按 [1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html) 更新。表格開始時通常是小亂數，不帶先驗語意；逐步調過後，相似任務中有用的關系才可能體現在向量裡。不是先把「貓很像狗」寫成標籤塞進去。

此時相同ID在任何位置仍查到同一個原始向量。兩次「看」放在不同句子中，先查出的向量一樣；當後面加入鄰字和位置資訊，兩個位置的中間表示才可能不同。要區分字的初始embedding與經過上下文混合的表示，別因某層輸出不同就以為查表也變了。

練習只把輸入第二格ID2改成ID1，先預測三個位置的向量都會相同，再執行核對形狀仍 `[1,3,3]`、三格均 `[1,0,0]`。這次只改查哪列，沒有改表；下一節會保留這三個位置的順序，讓它們一起提供預測資訊。
